In [1]:
import os
import django

os.environ.setdefault("DJANGO_SETTINGS_MODULE", "protwis.settings") 
django.setup()

from build.management.commands.PDB_sequence_helper import *

# construct_structure_annotation_override and fetch_pdb_info
from construct.functions import construct_structure_annotation_override, fetch_pdb_info
from django.shortcuts import get_object_or_404
from protein.models import Protein
from structure.models import Structure
from residue.models import Residue
from common.models import WebLink, WebResource

In [2]:
#Get structure from pdb code
def get_structures_by_pdb_code(pdb_code):
    """
    Retrieve Structure instances associated with a given PDB code.

    Args:
        pdb_code (str): The PDB code (e.g., "1ABC").

    Returns:
        QuerySet: A Django QuerySet containing matching Structure instances.
    """
    try:
        # Fetch the WebResource for PDB
        web_resource = WebResource.objects.get(slug='pdb')

        # Fetch the corresponding WebLink
        weblink = WebLink.objects.get(web_resource=web_resource, index=pdb_code.upper())

        # Retrieve associated Structure instances
        structures = Structure.objects.filter(pdb_code=weblink)

        return structures

    except WebResource.DoesNotExist:
        print("WebResource with slug 'pdb' does not exist.")
        return Structure.objects.none()
    except WebLink.DoesNotExist:
        print(f"No WebLink found for PDB code {pdb_code}.")
        return Structure.objects.none()

def get_wild_type_sequence_for_structure(pdb_code, structure_id):


    # 1. Fetch the Structure and its parent protein
    structure = get_object_or_404(Structure, id=structure_id)

    parent_protein = structure.protein_conformation.protein.parent

    if not parent_protein:
        return ""

    d = fetch_pdb_info(pdb_code, parent_protein, preferred_chain=structure.preferred_chain)
    entry_name = d['construct_crystal']['uniprot']
    preferred_chain=structure.preferred_chain
    deletions = []
    if 'deletions' in d:
        for del_range in d['deletions']:
            if del_range['start']==146 and structure.pdb_code.index=='4K5Y':
                #Manual fix for faulty 4K5Y annotation
                continue
            for i in range(del_range['start'],del_range['end']+1):
                deletions.append(i)
        #print("Annotation missing WT residues",d['deletions'])
    removed = []
    ## Remove segments that arent receptor (tags, fusion etc)
    if 'xml_segments' in d:
        for seg in d['xml_segments']:
            if seg[1]:
                # Odd rules to fit everything..
                # print(seg[1][0], entry_name)
                if seg[1][0]!=entry_name and seg[-1]!=True and seg[1][0]!='Uncharacterized protein' and 'receptor' not in seg[1][0]:
                    if seg[0].split("_")[1]==preferred_chain:
                        #print(seg[2],seg[3]+1)
                        #for i in range(seg[2],seg[3]+1):
                        # print(seg)
                        for i in seg[6]:
                            removed.append(i)
    # Reset removed, since it causes more problems than not

    removed, deletions = construct_structure_annotation_override(structure.pdb_code.index, removed, deletions)

    if len(deletions)>len(d['wt_seq'])*0.9:
        #if too many deletions
        removed = []
        deletions = []

    # 2. Build a queryset of residues that are NOT in the 'deletions'
    parent_residues = (
        Residue.objects
        .filter(protein_conformation__protein=parent_protein)
        .exclude(sequence_number__in=deletions)
        .order_by('sequence_number')
    )

    # 3. Concatenate the amino acids from that queryset
    parent_seq = "".join(res.amino_acid for res in parent_residues)

    return parent_seq


def gather_sequences_and_distances(pdb_code):
    """
    1) Find the first Structure matching pdb_code
    2) Identify its preferred chain
    3) Build the wild-type sequence
    4) Build the PDB sequence & CA-CA distances
    5) Return (wt_seq, pdb_seq, distances, structure, preferred_chain)
    """
    # Get the structure
    structures = get_structures_by_pdb_code(pdb_code)
    if not structures.exists():
        raise ValueError(f"No Structure found for PDB code {pdb_code}")
    structure = structures.first()
    
    # Get the chain
    preferred_chain = structure.preferred_chain or 'A'
    if ',' in preferred_chain:
        preferred_chain = preferred_chain.split(',')[0].strip()

    # Get the wild-type sequence (your existing function)
    wt_seq = get_wild_type_sequence_for_structure(pdb_code, structure.id)

    # Get the raw PDB text from DB
    pdb_text = structure.pdb_data.pdb
    if not pdb_text:
        raise ValueError(f"No PDB text stored for Structure {structure.id} ({pdb_code})")

    # Now call your helper function from PDB_sequence2_helper
    # to get the PDB sequence & distances
    pdb_seq, distances = generate_seq_and_distances_from_pdb_text(pdb_text, preferred_chain)

    return wt_seq, pdb_seq, distances, structure, preferred_chain


def see_results(pdb_code):
    wt_seq, pdb_seq, distances, structure, chain = gather_sequences_and_distances(pdb_code)

    print(f"WT seq length:  {len(wt_seq)}")
    print(f"PDB seq length: {len(pdb_seq)}")
    print(f"Distance list:  {len(distances)} elements")


    outlier_indexes = distances_stats(distances)

    # Then align:
    ref_seq, temp_seq, pdb_map = run_pairwisealigner(pdb_code, wt_seq, pdb_seq)

    # Then detect misalignment:
    detect_alignment_mistakes_and_reposition(
        pdb_code,
        wt_seq, 
        pdb_seq, 
        ref_seq, 
        temp_seq, 
        pdb_map, 
        distances, 
        outlier_indexes, 
        aanumber=3  # or however many residues you want to look back
    )




In [3]:
pdb_codes = ['7XJJ', '7F8V', '8TB7', '8X79', '8FU6', '8FMZ', '8GTI', '7TS0', '7T10', '6WHA', '8UWL', '8IW4', '8HAF', '8ZSJ', '8H0P', '4L6R', '8GTG', '8XQP', '3V2W', '8W8S', '7VVO', '9JR3', '6LN2', '1GZM', '7YFC', '6RZ5', '7EO4', '7B6W', '6KUX', '6NBI', '7S0F', '6KK1', '6K41', '7X8S', '8UXV', '8XWP', '6ZFZ', '8JWY', '8WVV', '8ID4', '7KI0', '6KJV', '8IWE', '7NA7', '6M1H', '7W6P', '8J23', '8YN4', '8HTI', '8XQO', '8FLQ', '8IRU', '8HN8', '9JR2', '8W8Q', '7PP1', '8WPG', '7WUJ', '8KH5', '6X18', '7KH0', '7SRS', '7EJK', '8JRV', '8JD1', '6NBF', '8W77', '8WKY', '6W25', '6NBH', '8IW1', '7VVK', '5VEX', '8ZFJ', '7VVJ', '8IRS', '3V2Y', '5VEW', '7DUQ', '8TR2', '5WIU', '7NA8', '6TPK', '8YW4', '7T8X', '8HOC', '7W7E', '7SIM', '8W8R', '6LPB', '6ZA8', '7UL2', '7RA3', '7C4S', '8FLS', '6PWC', '7KI1', '3SN6', '8HAO', '7FIY', '8TRD', '7RTB', '7SK5', '7T11', '7EWP', '7VVN', '8GGP', '3C9L', '8QW4', '7F8W', '7UL3', '6DO1', '8UXY', '6KK7', '7ZBE', '8SZF', '8TRC', '8WU1', '6Z4Q', '8A6C', '7EWR', '8YW5', '6KUW', '8TZQ', '8FLU', '7BB6', '6WHC', '5T1A', '8G94', '8YW3', '8PJK', '6ZG9', '7SBF', '8IKH', '7MTQ', '8SZI', '4PHU', '5ZKP', '7SIN', '8JCX', '2YCW', '7VQX', '7YMJ', '7FD9', '8JXW', '7EJA', '5WIV', '6U1N', '5UEN', '8T3Q', '4K5Y', '7M3J', '7WBJ', '7Y36', '7EJ8', '7RBT', '8GTM', '4RWD', '8J22', '7X8R', '7WU9', '7EZC', '7VVL', '8DZS', '8JCV', '7W57', '8YN2', '8J24', '7WIH', '8GGB', '8X7A', '8JWZ', '7SIL', '7VIH', '7EPT', '7UL5', '7RAN', '8V6U', '8GGA', '8HA0', '6ZG4', '8U02', '5JQH', '7EJ0', '9AVL', '8FLR', '8QJ2', '7SHF', '7C61', '8WCB', '8GGE', '7JVR', '8J46', '7VVM', '6KUY', '7Y35']

for pdb_code in pdb_codes:
    see_results(pdb_code)

WT seq length:  349
PDB seq length: 269
Distance list:  269 elements
Mean of all distances: 3.8712
Standard deviation of all distances: 0.7102
Lower bound for normal values: 1.7404
Upper bound for normal values: 6.0019
Filtered mean (after removing true outliers): 3.8016
Filtered variance: 0.0006
Filtered standard deviation: 0.0241
True outliers: [12.556099919374839, 11.01201507967285, 6.485127371280845]
Outlier at position 27
Outlier at position 101
Outlier at position 134

=== Detecting alignment mistakes for 7XJJ ===
Number of outliers: 3

Outlier at raw PDB index 27 (aligned pos 67)
Seq after the gap (temp_seq): 'STTNLFILNL'
Seq after the gap (pdb_seq):  'STTNLFILNL'
  No suspicious gap found nearby.

Outlier at raw PDB index 101 (aligned pos 147)
Seq after the gap (temp_seq): 'VSRNALLGVG'
Seq after the gap (pdb_seq):  'VSRNALLGVG'
  Possible mispositioned block (length=1): 'R'
Gap found at alignment pos 140 (length=6 dashes)
Alignment context before gap (temp_seq):           VDRYV